# 캐시 라인과 메모리 접근 패턴 - CPU 캐시와 행렬 연산 최적화

- Tutorial ID: `ull-4`
- Tutorial: 캐시 라인과 메모리 접근 패턴
- Section ID: `ull-4-1`
- Section: CPU 캐시와 행렬 연산 최적화


In [ ]:
# ============================================================
# 코드 읽는 법 — CPU 캐시와 행렬 연산 최적화
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) autoregressive decoding에서 이전 K/V를 재사용해 계산을 줄이는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
import time

print("=" * 62)
print("캐시 라인과 메모리 접근 패턴")
print("=" * 62)

In [ ]:
# 1. 캐시 라인 기초

In [ ]:
print("\n1. 캐시 라인 기초")
print("-" * 50)

cache_line_bytes = 64
float32_bytes = 4
floats_per_line = cache_line_bytes // float32_bytes

print(f"  캐시 라인 크기: {cache_line_bytes} bytes")
print(f"  float32 크기: {float32_bytes} bytes")
print(f"  캐시 라인당 float32: {floats_per_line}개")

# 배열의 메모리 레이아웃
arr = np.zeros((4, 4), dtype=np.float32)
print(f"\n  4×4 float32 배열:")
print(f"    총 크기: {arr.nbytes} bytes")
print(f"    캐시 라인 수: {arr.nbytes / cache_line_bytes}")
print(f"    한 행(4개 float) = {4 * 4} bytes < 64 bytes (한 캐시 라인)")

In [ ]:
# 2. 행 vs 열 순회 성능 차이

In [ ]:
print("\n2. 행 순회 vs 열 순회 성능")
print("-" * 50)

n = 200

# Row-major 배열
A = np.random.randn(n, n).astype(np.float32)

# 행 순회 (cache-friendly)
def row_traverse(arr):
    s = np.float32(0)
        for i in range(arr.shape[0]):
                for j in range(arr.shape[1]):
            s += arr[i, j]
    return s

# 열 순회 (cache-unfriendly)
def col_traverse(arr):
    s = np.float32(0)
        for j in range(arr.shape[1]):
                for i in range(arr.shape[0]):
            s += arr[i, j]
    return s

repeats = 3

start = time.perf_counter()
for _ in range(repeats):
    row_traverse(A)
row_time = (time.perf_counter() - start) / repeats

start = time.perf_counter()
for _ in range(repeats):
    col_traverse(A)
col_time = (time.perf_counter() - start) / repeats

print(f"  {n}×{n} 배열 순회:")
print(f"    행 순회: {row_time*1000:.1f} ms")
print(f"    열 순회: {col_time*1000:.1f} ms")
print(f"    비율: {col_time/row_time:.1f}x 느림")

In [ ]:
# 3. 타일링 (Blocking) 최적화

In [ ]:
print("\n3. 행렬 곱 타일링 최적화")
print("-" * 50)

def naive_matmul(A, B):
    """순진한 3중 루프 행렬 곱"""
    M, K = A.shape
    K2, N = B.shape
    C = np.zeros((M, N), dtype=A.dtype)
        for i in range(M):
                for j in range(N):
                        for k in range(K):
                C[i, j] += A[i, k] * B[k, j]
    return C

def tiled_matmul(A, B, tile_size=16):
    """타일링된 행렬 곱"""
    M, K = A.shape
    K2, N = B.shape
    C = np.zeros((M, N), dtype=A.dtype)
    
        for ii in range(0, M, tile_size):
                for jj in range(0, N, tile_size):
                        for kk in range(0, K, tile_size):
                # 작은 타일끼리 곱 (캐시에 맞음)
                i_end = min(ii + tile_size, M)
                j_end = min(jj + tile_size, N)
                k_end = min(kk + tile_size, K)
                
                                for i in range(ii, i_end):
                                        for j in range(jj, j_end):
                                                for k in range(kk, k_end):
                            C[i, j] += A[i, k] * B[k, j]
    return C

n = 64  # 작은 크기로 시연

A = np.random.randn(n, n).astype(np.float32)
B = np.random.randn(n, n).astype(np.float32)

start = time.perf_counter()
C_naive = naive_matmul(A, B)
t_naive = time.perf_counter() - start

start = time.perf_counter()
C_tiled = tiled_matmul(A, B, tile_size=16)
t_tiled = time.perf_counter() - start

start = time.perf_counter()
C_numpy = A @ B
t_numpy = time.perf_counter() - start

print(f"  {n}×{n} 행렬 곱:")
print(f"    순진한 3중 루프: {t_naive*1000:.1f} ms")
print(f"    타일링 (16×16): {t_tiled*1000:.1f} ms")
print(f"    NumPy (BLAS):    {t_numpy*1000:.3f} ms")
print(f"\n  BLAS 대비 순진한 구현: {t_naive/t_numpy:.0f}x 느림")

print(f"\n  타일링 효과:")
print(f"    16×16 타일 = 16×16×4 = 1,024 bytes")
print(f"    L1 캐시 (32KB)에 약 32개 타일 수용 가능")
print(f"    → 데이터 재사용 극대화!")

In [ ]:
# 4. GPU 메모리 계층 비교

In [ ]:
print("\n4. GPU vs CPU 메모리 계층")
print("-" * 50)

print("""
  CPU:                          GPU (A100):
  ────────────                  ────────────
  레지스터  ~1ns    ~1KB        레지스터  ~1ns    ~20MB
  L1 캐시   ~1ns    32KB       공유메모리 ~5ns   192KB/SM
  L2 캐시   ~4ns    256KB      L2 캐시   ~20ns   40MB
  L3 캐시   ~12ns   32MB       HBM       ~200ns  80GB
  DRAM      ~60ns   ~GB        
  
  핵심 차이:
  - GPU는 캐시가 작지만 레지스터가 매우 많음
  - GPU의 공유 메모리 = 프로그래머가 직접 관리하는 캐시
  - FlashAttention은 공유 메모리를 최대한 활용
""")

In [ ]:
# 5. NumPy의 스트라이드와 뷰

In [ ]:
print("5. 스트라이드와 메모리 뷰")
print("-" * 50)

arr = np.arange(12, dtype=np.float32).reshape(3, 4)
print(f"  원본 (3×4, C-contiguous):")
print(f"    strides: {arr.strides} bytes")
print(f"    데이터: {arr.ravel()}")

# 전치 = 스트라이드만 교환 (복사 없음!)
arr_T = arr.T
print(f"\n  전치 (4×3, Non-contiguous):")
print(f"    strides: {arr_T.strides} bytes")
print(f"    같은 메모리를 공유: {np.shares_memory(arr, arr_T)}")
print(f"    C-contiguous: {arr_T.flags['C_CONTIGUOUS']}")

# 연속화
arr_T_contig = np.ascontiguousarray(arr_T)
print(f"\n  연속화 후:")
print(f"    strides: {arr_T_contig.strides} bytes")
print(f"    같은 메모리: {np.shares_memory(arr, arr_T_contig)}")
print(f"    → 실제 데이터 복사 발생! (추가 메모리 사용)")